In [1]:
from _lib import *
from data import *

- load all the historical data and universe

In [2]:
boa = load_star_board()
tickers = boa["ticker"].tolist()
qt.log.info(f"no of tickers on STAR board - [{len(tickers)}]")

[JUSTY.LOG]	2026-05-09 19:54:51,591 - qt.common.help - INFO - no of tickers on STAR board - [609]


- rebalance parameters

In [3]:
eff_date = dt.date(2026, 6, 13)
annc_date = dt.date(2026, 5, 30)
cutoff_date = dt.date(2026, 4, 30)
hist_start = dt.date(2025, 5, 1)
cutoff_10 = (cutoff_date + pd.offsets.BDay(10)).date()

In [4]:
sse_holidays = load_sse_holidays()

/home/justy/private/INTERVIEWS/ap_star50/data.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sse_holidays_t['date'] = pd.to_datetime(sse_holidays_t['date']).dt.date


- get listing date

In [5]:
history = load_historical_data_ohlcv(tickers, hist_start, cutoff_date)

- add scraped data from sse

In [6]:
# merge info_df and tu on ticker
t_ld = get_listing_dates(tickers)

# add listing date to star board
boa = pd.merge(boa, t_ld, on="ticker", how="outer")

- special treatment securities

In [7]:
st_securities = boa[boa['st']]['ticker'].tolist()
qt.log.info(f"[{len(st_securities)}] ST securities: {st_securities}")

[JUSTY.LOG]	2026-05-09 19:54:56,050 - qt.common.help - INFO - [12] ST securities: ['688022', '688033', '688053', '688066', '688076', '688184', '688201', '688270', '688287', '688496', '688622', '688646']


- eligibility
	- Listing time > 6 months. 
		1. If no of securities listed > 12 months is b/w 100 to 150 then requirement changes to > 12 months
	- For securities with daily avg total market_cap since initial listing in top 5, listing time should be > 3 months as of 10th trading days after end date of data (cutoff date)
	- For securities with daily avg total market_cap since initial listing in top 3, listint time should be > 1 month
	- Non-* ST securities
	- No violation of laws/reg, no financial problems etc

In [8]:
history_after_cof = history.copy()

In [22]:
univ = boa[['ticker', 'listing_date', 'name', 'market_cap', 'st', 'shares_total', 'shares_tradable']].copy()
univ = univ[~univ['st']].reset_index(drop=True)
univ['month_to_cof'] = univ['listing_date'].apply(lambda d: (cutoff_date.year - d.year) * 12 + (cutoff_date.month - d.month)  - int(cutoff_date.day < d.day))
univ['month_to_cof10'] = univ['listing_date'].apply(lambda d: (cutoff_10.year - d.year) * 12 + (cutoff_10.month - d.month)  - int(cutoff_10.day < d.day))

# add shares outstanding from yfinance
t_yf = get_yf_info(tickers=univ['ticker'].tolist())
univ = pd.merge(univ, t_yf[['ticker', 'shs_os_yf', 'shs_ff_yf']], on='ticker', how='left')

# if shs_os_yf is not null, use it as shs_total, otherwise use shares_total
# univ.loc[univ['shs_os_yf'] != 0, 'shs_total'] = univ['shs_os_yf']
# univ.loc[univ['shs_os_yf'] == 0, 'shs_total'] = univ['shares_total']
univ['shs_for_avg'] = univ['shares_total']

# add avg total mcap and avg value traded
avg_val_traded_n_mcap = get_avg_vol_mcap(history, univ, tickers=univ['ticker'].unique().tolist(), shs_col='shs_for_avg')

univ = pd.merge(univ, avg_val_traded_n_mcap, on='ticker', how='left')
univ['tmcap_rank'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ['vtrad_rank'] = univ['avg_val_traded'].rank(ascending=False, method='min')

# read weight of existing index
curr_s50 = load_star50_weights()

# add weight column
univ = pd.merge(univ, curr_s50[['ticker', 'curr_weight']], on='ticker', how='left')

- add eligibility

In [23]:
no_of_securities_mt_12m = len(univ[univ['month_to_cof'] >= 12])
listing_month_cutoff = 12 if no_of_securities_mt_12m > 100 else 6
qt.log.info(f"Listing month cutoff: {listing_month_cutoff} months (securities with month_to_cof >= {listing_month_cutoff}: {no_of_securities_mt_12m})")

univ['listing_elig'] = (
	((univ['tmcap_rank'] <= 3) & (univ['month_to_cof'] >= 1)) |
	((univ['tmcap_rank'] <= 5) & (univ['month_to_cof10'] >= 3)) |
	(univ['month_to_cof'] >= listing_month_cutoff)
)

univ = univ[univ['listing_elig']].reset_index(drop=True)
univ['vtrad_rank2'] = univ['avg_val_traded'].rank(ascending=False, method='min')

# drop 10% stocks based on avg value traded rank
univ = univ[univ['vtrad_rank2'] <= 0.9*len(univ)].reset_index(drop=True)
univ['tmcap_rank2'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ = univ.sort_values('tmcap_rank2').reset_index(drop=True)

[JUSTY.LOG]	2026-05-09 20:15:53,596 - qt.common.help - INFO - Listing month cutoff: 12 months (securities with month_to_cof >= 12: 558)


In [24]:
univ[
	(univ['tmcap_rank2'] <= 40) &
	(univ['curr_weight'].isna())
]
univ[
	(univ['tmcap_rank2'] > 60) &
	(~univ['curr_weight'].isna())
]

,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_ff_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
3,688235,2021-12-15,百济神州,381.54,False,"1,542.00",115.00,52,52,115.06,80.76,"1,542.00",405.57,898.73,242,4.00,44.00,NaN,True,36.00,4.00
4,688795,2025-12-05,摩尔线程,330.43,False,470.00,29.00,4,5,470.03,214.92,470.00,288.23,"2,380.07",96,5.00,11.00,NaN,True,9.00,5.00
5,688347,2023-08-07,华虹公司,276.21,False,"1,738.00",408.00,32,33,407.75,891.90,"1,738.00",167.39,"2,089.95",232,8.00,14.00,NaN,True,12.00,6.00
24,688385,2021-08-04,复旦微电,59.04,False,824.00,539.00,56,57,539.39,575.01,824.00,51.11,858.31,242,35.00,48.00,NaN,True,40.00,25.00
25,688331,2022-03-31,荣昌生物,69.94,False,564.00,163.00,48,49,354.93,348.58,564.00,50.92,765.58,242,36.00,61.00,NaN,True,52.00,26.00
30,688498,2022-12-21,源杰科技,135.37,False,86.00,85.00,40,40,85.95,59.89,86.00,46.61,"2,104.42",242,41.00,13.00,NaN,True,11.00,31.00
32,688428,2022-09-21,诺诚健华,46.50,False,"1,765.00",268.00,43,43,268.36,"1,171.79","1,765.00",44.61,233.12,242,44.00,235.00,NaN,True,207.00,33.00
38,688110,2021-12-10,东芯股份,68.47,False,442.00,442.00,52,53,442.25,282.80,442.00,41.25,"2,254.12",242,50.00,12.00,NaN,True,10.00,39.00


,ticker,listing_date,name,market_cap,st,shares_total,shares_tradable,month_to_cof,month_to_cof10,shs_os_yf,shs_ff_yf,shs_for_avg,avg_total_mcap,avg_val_traded,count_data_pts,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
60,688297,2022-06-29,中无人机,32.14,False,675.00,675.00,46,46,675.00,211.81,675.00,33.76,633.28,242,74.00,83.00,0.50,True,70.00,61.00
64,688349,2022-06-22,三一重能,30.53,False,"1,226.00","1,189.00",46,46,"1,226.40",242.45,"1,226.00",32.17,126.24,242,78.00,376.00,0.24,True,341.00,65.00
66,688278,2020-01-17,特宝生物,23.33,False,408.00,408.00,75,75,408.19,161.56,408.00,30.94,188.44,242,80.00,281.00,0.36,True,249.00,67.00
76,688114,2022-09-09,华大智造,21.95,False,417.00,414.00,43,44,416.52,172.23,417.00,27.00,251.85,242,93.00,216.00,0.51,True,188.00,77.00


In [ ]:
univ[univ['ticker'].isin(
	[
		'688498', '688110', '688002',
		'688114', '688278', '688349'	
  	]
)]

In [ ]:
excls = ['688220', '688301', '688385']
incls = ['688213', '688278', '688578']
res_list = ["688608", "688425", "688361", "688568", "688172",]
univ[univ['ticker'].isin(excls)]
univ[univ['ticker'].isin(incls)]
univ[univ['ticker'].isin(res_list)]